In [ ]:

# x = df['highway-mpg']
# y = df['price']
f1 = np.polyfit(x, y, 11)
p1 = np.poly1d(f1)
print(p1)
PlotPolly(p1,x,y, 'Highway MPG')


# 从 scikit-learn 的数据预处理模块 sklearn.preprocessing 中导入 PolynomialFeatures 类。
from sklearn.preprocessing import PolynomialFeatures


# 从 sklearn.preprocessing 实例化一个多项式特征生成器，degree=2 表示生成最高 2 次的多项式组合。
# 此处只是创建对象，还没有真正处理数据（惰性构建，需要后续 fit / fit_transform 才会计算）。
pr=PolynomialFeatures(degree=2)  #标准化
pr

# fit_transform(Z) — 实际上是两个动作的组合：
# fit(Z)：扫描 Z，统计每个特征的取值范围、组合方式，确定要生成哪些多项式项（哪些一次项、平方项、交叉项）。
# transform(Z)：按拟合好的规则实际计算并输出新特征矩阵。
# 两者合并写成 fit_transform 是 sklearn Transformer 的标准接口习惯（一次性完成“学习规则 + 应用规则”）。
Z_pr=pr.fit_transform(Z)

# 对 4 个特征做 2 阶展开（include_bias=True 默认）：
# 1 个截距列（全 1）
# 4 个一次项
# 4 个平方项 horsepower 的平方  curb-weight 的平方 engine-size 的平方 highway-mpg 的平方
# 6 个交叉项 horsepower × curb-weight  horsepower × engine-size .....
#也就是使用了fit_transform这个方法，能够对原来的四个列进行升维，新增11列
# 它们对应升维后矩阵中的 4 个一次项列——这 4 列是“原有的”。升维后变成 15 列，新增的 11 列就是：1 个截距列 + 4 个平方项 + 6 个交叉项。
#特征指数 powers_ 记录的是 Z_pr 每一列的“指数配方”
# powers_ =（powers_ 是"列清单"的转置视图——每行是一个列的配方，所以 11 个新增列在）
# [[0 0 0 0]   ①  1（截距列，全1）          [新增]
#  [1 0 0 0]   ②  x1 = horsepower          （原有）
#  [0 1 0 0]   ③  x2 = curb-weight         （原有）
#  [0 0 1 0]   ④  x3 = engine-size         （原有）
#  [0 0 0 1]   ⑤  x4 = highway-mpg         （原有）
#  [2 0 0 0]   ⑥  x1² = horsepower²        [新增]
#  [1 1 0 0]   ⑦  x1·x2                    [新增]
#  [1 0 1 0]   ⑧  x1·x3                    [新增]
#  [1 0 0 1]   ⑨  x1·x4                    [新增]
#  [0 2 0 0]   ⑩  x2²                      [新增]
#  [0 1 1 0]   ⑪  x2·x3                    [新增]
#  [0 1 0 1]   ⑫  x2·x4                    [新增]
#  [0 0 2 0]   ⑬  x3²                      [新增]
#  [0 0 1 1]   ⑭  x3·x4                    [新增]
#  [0 0 0 2]]  ⑮  x4²                      [新增]



# Z = df[['horsepower', 'curb-weight', 'engine-size', 'highway-mpg']]
# .shape 是 pandas DataFrame 的属性（不是函数，所以不带括号），返回一个元组 (行数, 列数)，即 (样本数, 特征数)。
# (201, 4)
# 201：数据集中有效样本数（去掉含缺失值的价格等行之后）；
# 4：horsepower、curb-weight、engine-size、highway-mpg 这 4 个自变量。
Z.shape

# 和上面的进行对比
# PolynomialFeatures(degree=2) 把特征列从 4 列扩到了 15 列，而样本数保持 201 不变。如果忘记对比，容易在后面 lm.fit(Z_pr, y) 时误以为维数变化正常或数据出错。
Z_pr.shape

# 1	x0	x1	x2	x3	x0²	x0·x1	x0·x2	x0·x3	x1²	x1·x2	x1·x3	x2²	x2·x3	x3²
# 1	111	2548	130	27	12321	282828	14430	2997	6492304	331240	68796	16900	3510	729
# 1	111	2548	130	27	12321	282828	14430	2997	6492304	331240	68796	16900	3510	729
# 1	154	2823	152	26	23716	434742	23408	4004	7969329	429096	73398	23104	3952	676
# 例如第 7 列 x0·x1（horsepower × curb-weight）


# Pipeline 是 scikit-learn 的工作流编排工具，作用是把多个数据处理和建模步骤按顺序串成一条链，对外表现得像一个模型。
from sklearn.pipeline import Pipeline
# StandardScaler 是标准化器，把每个特征变换为均值 0、标准差 1
from sklearn.preprocessing import StandardScaler
# 两个导入对应的就是这条流水线中“数据预处理”环节的两个角色：一个负责统一量纲（StandardScaler），一个负责按序编排（Pipeline）。

# 这是为下一步创建 Pipeline 准备的步骤清单——一个由 3 个 (名称, 对象) 元组组成的 Python 列表，每个元组代表流水线中的一站。
# ① ('scale', StandardScaler()) — 第一站：标准化 把每个特征变换为均值 0、标准差 1
#必须放在最前面，原因你在 Z_pr 的输出里已经见过：原始 4 列量级悬殊（highway-mpg 只有二三十，curb-weight 几千），若先升维，数值不稳定，所以我们要先标准化，再升维
# ② ('polynomial', PolynomialFeatures(include_bias=False)) — 第二站：多项式升维（这里没有指定 degree，默认 degree=2，效果与之前 pr = PolynomialFeatures(degree=2) 相同（4 列 → 14 列）。）
# 关键区别：include_bias=False。之前单独用时默认含偏置（15 列，多一列全 1）；这里去掉它，因为第三站的 LinearRegression 自带截距项，不需要数据里再提供一列 1——避免冗余。我们给数据人为加一列“恒等于 1 的特征”；全 1 列只是一个取值恒为 1 的“假特征”，为了体现b0*1这一项
# ③ ('model', LinearRegression()) — 第三站（终点）：线性回归 升维后的 14 列特征交给线性回归拟合，整体上等效于多项式回归：
Input=[('scale',StandardScaler()), ('polynomial', PolynomialFeatures(include_bias=False)), ('model',LinearRegression())]
# 前面的做法（第 16–23 行）	本行的 Pipeline 做法
# 流程	    手动 fit_transform → 手动 lm.fit	一条链自动完成
# 偏置列	有（15 列，含冗余全 1 列）	无（14 列，include_bias=False）
# 标准化	没有（升维后数值巨大）	有（先缩放再升维）
# 预测一致性	需手动重复每个预处理步骤	自动保证

# 此处只是组装，还没有训练——三个子对象仍然是未拟合状态，pipe 此刻只是一个“执行计划”。
pipe=Pipeline(Input)
pipe #Pipeline(steps=[('scale', StandardScaler()), ('polynomial', PolynomialFeatures(include_bias=False)),('model', LinearRegression())])

# 数据清洗 把 Z（horsepower、curb-weight、engine-size、highway-mpg 4 列）的 dtype 统一转成 float64。
# 为什么需要这一步？ 因为 usedcars.csv 原始数据里部分数值列是以字符串（object）类型读入的（历史原因：该数据集的 horsepower、normalized-losses 等列含 ? 缺失标记，被 pandas 当作 object 处理；或从其他 notebook 传递过来时保留了 object 类型）。
Z = Z.astype(float)
# pipe.fit(Z, y) — 流水线训练 一行代码触发整条链，内部按 Input 定义的顺序自动执行三步：
# pipe.fit(Z, y)
#    ├─ ① StandardScaler.fit(Z) / .transform(Z)
#    │      学习每个特征的 μ 和 σ，把 Z 变成均值0、标准差1 → (201, 4)
#    ├─ ② PolynomialFeatures.fit / .transform
#    │      对标准化后的数据做 2 阶展开（include_bias=False）→ (201, 14)
#    └─ ③ LinearRegression.fit(升维后的特征, y)
#           用最小二乘法求出 β，训练完成
pipe.fit(Z,y)

# 用算出来的式子开始预测价格
ypipe=pipe.predict(Z)
ypipe[0:4]
# ypipe[0:4] 就是模型对前 4 辆车预测价格的浮点数组，前两项因车辆配置相同而相等；用它对照 y[0:4] 的真实价格，可以直观感受模型预测与实际值之间的差距



# 总结一下：手动挡与自动档（到fit计算系数之前，全自动化，标准化，升维，传递模型，用最小二乘法拟合一个线性模型，在 Lab01 里，它是所有模型（简单线性、多元线性、多项式回归）的核心引擎。）
# 手动
# 第1步：自己算标准化
# scaler = StandardScaler()
# Z_scaled = scaler.fit_transform(Z)     # 自己拿中间结果

# # 第2步：自己传给升维器
# poly = PolynomialFeatures(degree=2, include_bias=False)
# Z_poly = poly.fit_transform(Z_scaled)  # 又自己拿一次

# # 第3步：自己传给模型
# lm = LinearRegression()
# lm.fit(Z_poly, y)                      # 自己又拿一次

# # 预测时：必须手动重复整条链！
# Z_new_scaled = scaler.transform(Z_new)     # 忘一步就出错
# Z_new_poly = poly.transform(Z_new_scaled)
# y_pred = lm.predict(Z_new_poly)

# 自动
# pipe = Pipeline(Input)
# pipe.fit(Z, y)                # 训练：内部自动串联三步
# y_pred = pipe.predict(Z_new)  # 预测：内部自动重复整条链

